# Traffic Law V2 - Eval Overview\n
\n
Notebook này đọc kết quả từ `scripts/run_eval_overview.py` để phục vụ báo cáo.\n
\n
Metrics chính:\n
- BERTScore: Precision / Recall / F1 nếu package `bert-score` đã được cài.\n
- RAG Triad heuristic: context relevance, groundedness, answer relevance.\n
- Expected term/citation coverage để bắt lỗi pháp lý cơ bản.\n

## 1. Chạy eval script trước\n
\n
```bash\n
cd /Users/m1/Documents/rag-thesis1/traffic-law-v2\n
PYTHONPATH=src .venv/bin/python scripts/run_eval_overview.py \\\n
  --dataset eval/datasets/traffic_law_eval.jsonl \\\n
  --index-dir data/index \\\n
  --out-dir eval/outputs\n
```\n
\n
Nếu chỉ muốn test pipeline không gọi LLM:\n
\n
```bash\n
PYTHONPATH=src .venv/bin/python scripts/run_eval_overview.py --skip-generation\n
```\n

In [2]:
from pathlib import Path
import json

RESULTS = Path('../outputs/eval_overview_results.jsonl')
SUMMARY = Path('../outputs/eval_overview_summary.json')

rows = [json.loads(line) for line in RESULTS.read_text(encoding='utf-8').splitlines() if line.strip()]
summary = json.loads(SUMMARY.read_text(encoding='utf-8'))
len(rows), summary['summary']['averages']

(20,
 {'rag_context_relevance': 0.6095,
  'rag_groundedness': 0.9373,
  'rag_answer_relevance': 1.0,
  'rag_triad_mean': 0.8489,
  'bertscore_precision': None,
  'bertscore_recall': None,
  'bertscore_f1': None,
  'answer_expected_term_coverage': 0.9667,
  'context_expected_term_coverage': 0.8758})

In [5]:
# Optional pandas view. Nếu pandas chưa cài, cell này sẽ dùng list/dict thường.\n
try:
    import pandas as pd
    df = pd.DataFrame([
        {
            'id': r['id'],
            'category': r['category'],
            'rag_context_relevance': r['rag_triad']['context_relevance'],
            'rag_groundedness': r['rag_triad']['groundedness'],
            'rag_answer_relevance': r['rag_triad']['answer_relevance'],
            'rag_triad_mean': r['rag_triad']['triad_mean'],
            'bertscore_p': r.get('bertscore', {}).get('precision'),
            'bertscore_r': r.get('bertscore', {}).get('recall'),
            'bertscore_f1': r.get('bertscore', {}).get('f1'),
            'answer_term_coverage': r['expected_checks']['answer_expected_terms']['score'],
            'context_term_coverage': r['expected_checks']['context_expected_terms']['score'],
            'fallback': r['answer_meta']['fallback'],
        } for r in rows
    ])
    display(df.head(20))
except Exception as exc:
    print('pandas unavailable:', exc)
    for r in rows[:5]:
        print(r['id'], r['category'], r['rag_triad'])

,id,category,rag_context_relevance,rag_groundedness,rag_answer_relevance,rag_triad_mean,bertscore_p,bertscore_r,bertscore_f1,answer_term_coverage,context_term_coverage,fallback
0,scope_nd100,scope,0.6579,1.0000,1.0,0.8860,None,None,None,1.0000,1.0000,False
1,definition_e_motorbike,definition,0.5704,1.0000,1.0,0.8568,None,None,None,1.0000,1.0000,False
2,household_penalty_subject,definition,0.6536,1.0000,1.0,0.8845,None,None,None,1.0000,1.0000,False
3,remedial_measure_sign,remedial_measure,0.6046,0.6800,1.0,0.7615,None,None,None,1.0000,0.2500,False
4,car_horn_22_5,penalty_car,0.5445,0.9667,1.0,0.8371,None,None,None,1.0000,0.8000,False
5,car_change_lane_no_signal,penalty_car,0.6522,0.9500,1.0,0.8674,None,None,None,1.0000,0.7500,False
6,car_phone_driving,penalty_car,0.6527,1.0000,1.0,0.8842,None,None,None,1.0000,1.0000,False
7,car_red_light,penalty_car,0.6533,1.0000,1.0,0.8844,None,None,None,1.0000,1.0000,False
8,car_wrong_way_expressway,penalty_car,0.6527,1.0000,1.0,0.8842,None,None,None,1.0000,1.0000,False
9,car_alcohol_over_80,penalty_car,0.5461,0.8929,1.0,0.8130,None,None,None,1.0000,0.5000,False


In [4]:
# Category breakdown\n
try:
    display(df.groupby('category')[['rag_triad_mean', 'answer_term_coverage', 'context_term_coverage']].mean().round(4))\n
except Exception as exc:
    print('category table unavailable:', exc)
    print(summary['summary']['by_category'])

SyntaxError: unexpected character after line continuation character (2768362451.py, line 3)

In [ ]:
# Charts for report\n
try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df['category'].value_counts().plot(kind='bar', ax=axes[0], title='Eval cases by category')
    df[['rag_context_relevance', 'rag_groundedness', 'rag_answer_relevance']].mean().plot(kind='bar', ax=axes[1], title='RAG Triad averages')\n
    plt.tight_layout()
except Exception as exc:
    print('matplotlib unavailable:', exc)

In [ ]:
# Lowest-score cases for error analysis\n
try:
    display(df.sort_values('rag_triad_mean').head(8))
except Exception:
    for r in sorted(rows, key=lambda x: x['rag_triad']['triad_mean'])[:8]:
        print(r['id'], r['rag_triad']['triad_mean'], r['query'])

In [ ]:
# Inspect one case in detail\n
case_id = rows[0]['id']
case = next(r for r in rows if r['id'] == case_id)
print('ID:', case['id'])
print('Query:', case['query'])
print('Reference:', case['reference_answer'])
print('Generated:', case['generated_answer'])
print('RAG Triad:', case['rag_triad'])
print('BERTScore:', case.get('bertscore'))
print('Expected checks:', case['expected_checks'])